In [1]:
import torch
import torch.nn as nn

B = 4       # Batch Size：批次大小，一次输入4个样本
S = 128     # Sequence Length：每个样本的token序列长度
H = 768     # Hidden Size：隐藏层维度，每个token的特征向量长度
V = 30522   # Vocabulary Size：词表大小，BERT-base词表规模，GPT类模型通常更大

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")

# ============================================================
# 1. 定义输出投影层（LM Head），对应公式 Y = XW
# ============================================================
# 注意：PyTorch的nn.Linear权重默认形状是 [out_features, in_features]，即 [V, H]
# 计算时内部执行的是 y = x @ weight.T + bias，和公式 Y=XW（W形状H×V）数学上完全等价
# bias=False 是简化场景，很多大模型输出层不加偏置
lm_head = nn.Linear(in_features=H, out_features=V, bias=False).to(device)

print(f"\n权重W的形状：{lm_head.weight.shape} [V, H] = [{V}, {H}]")


# ============================================================
# 第一问：形状是否匹配？验证矩阵乘法合法性
# ============================================================
print("\n===== 第一问：形状匹配验证 =====")
X = torch.randn(B, S, H).to(device)  # 输入张量X，形状[B, S, H]
print(f"输入张量X的形状：{X.shape} [B, S, H] = [{B}, {S}, {H}]")

# 工程上会把batch和序列维度合并，方便矩阵乘法计算
# 因为输出投影是按照逐token进行的，所以可以把输入张量X从[B, S, H] reshape成[B*S, H]
X_flat = X.reshape(B*S, H)  # 形状变为[B*S, H]
print(f"输入张量X_flat的形状：{X_flat.shape} [B*S, H] = [{B*S}, {H}]")

# 执行线性投影计算
Y_flat = lm_head(X_flat)  # 输出张量Y_flat，形状[B*S, V]
print(f"输出张量Y_flat的形状：{Y_flat.shape} [B*S, V] = [{B*S}, {V}]")

# 还原回三维形状，对应每个样本每个位置的词表得分
Y = Y_flat.reshape(B, S, V)  # 形状变为[B, S, V]
print(f"输出张量Y的形状：{Y.shape} [B, S, V] = [{B}, {S}, {V}]")

print("结论：内维H匹配，矩阵乘法合法，输出形状符合预期")


Using device: cuda
CUDA device name: NVIDIA GeForce RTX 5080

权重W的形状：torch.Size([30522, 768]) [V, H] = [30522, 768]

===== 第一问：形状匹配验证 =====
输入张量X的形状：torch.Size([4, 128, 768]) [B, S, H] = [4, 128, 768]
输入张量X_flat的形状：torch.Size([512, 768]) [B*S, H] = [512, 768]
输出张量Y_flat的形状：torch.Size([512, 30522]) [B*S, V] = [512, 30522]
输出张量Y的形状：torch.Size([4, 128, 30522]) [B, S, V] = [4, 128, 30522]
结论：内维H匹配，矩阵乘法合法，输出形状符合预期
